In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
admin_rdm_url = 'http://localhost:8001/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
idp_name_2 = 'FakeCAS'
idp_username_2 = None
idp_password_2 = None
weko_url = 'https://weko3.rdm.example.com/'
weko_admin_email = None
weko_admin_password = None
weko_institution_name = 'Massachusetts Institute of Technology [Test]'
weko_index_name = 'GRDM-Collaboration-Test-VR-1'
clear_apps = True
is_staff = False
default_result_path = None
close_on_fail = False
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False
project_name = None
oauth_application_name = None
project_url = None
default_storage_label = 'NII Storage'

In [ ]:
import asyncio
import importlib
import os
import shutil
import tempfile
from urllib.parse import urljoin, urlparse

if idp_username_2 is None:
    prompt = f"Username for {idp_name_2 or 'user account'}: "
    idp_username_2 = input(prompt)
if idp_password_2 is None:
    prompt = f"Password for {idp_username_2}: "
    idp_password_2 = getpass(prompt)

if idp_username_1 is None:
    prompt = f"Username for {idp_name_1 or 'admin account'}: "
    idp_username_1 = input(prompt)
if idp_password_1 is None:
    prompt = f"Password for {idp_username_1}: "
    idp_password_1 = getpass(prompt)

if weko_admin_email is None:
    weko_admin_email = input('WEKO admin email: ')
if weko_admin_password is None:
    weko_admin_password = getpass('WEKO admin password: ')
if weko_institution_name is None:
    weko_institution_name = input('機関名 (例: GakuNin RDM IdP): ')
if weko_index_name is None:
    weko_index_name = 'TestJ'

if project_name is None:
    project_name = datetime.now().strftime('TEST-WEKO-%Y%m%d%H%M')
if oauth_application_name is None:
    oauth_application_name = datetime.now().strftime('TEST-WEKO-APP-%Y%m%d%H%M')



# 機関でのJAIRO Cloud設定有効化

- サブシステム名: アドオン
- ページ/アドオン: WEKO
- 機能分類: アドオン管理
- シナリオ名: OAuthクライアント追加
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM一般ユーザー, WEKO管理者、ユーザーadmin: GRDM統合管理者)

## ウェブブラウザの新規プライベートウィンドウでGRDM統合管理者画面を表示する

GRDM統合管理者画面のログインページが表示されること

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir


In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)
    await expect(page.locator('//form[@id = "login-form"] | //div[contains(@class, "login-logo")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step, new_page=True)


## IdPを利用し、ユーザーadminとしてログインする

GRDM統合管理者画面が表示されること

In [ ]:
async def _step(page):
    await grdm.login_as_admin(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await expect(page.locator('//a[@href = "/addons/"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## アドオン利用制御をクリックする

機関一覧が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/addons/"]').click()
    if is_staff:
        await expect(page.locator('//h4/span[text() = "JAIRO Cloud"]')).to_be_visible(timeout=transition_timeout)
    else:
        await expect(page.locator('//input[@type = "search"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (統合管理者の場合) テストユーザーの所属機関をクリックする

アドオン利用制御画面に JAIRO Cloud が表示されること

In [ ]:
async def _step(page):
    if is_staff:
        await expect(page.locator('//h4/span[text() = "JAIRO Cloud"]')).to_be_visible(timeout=transition_timeout)
        return
    await page.locator('//input[@type = "search"]').fill(weko_institution_name)
    institution_link = page.locator(f'//a[contains(text(), "{weko_institution_name}")]')
    await expect(institution_link).to_be_visible(timeout=transition_timeout)
    await institution_link.click()
    await expect(page.locator('//h4/span[text() = "JAIRO Cloud"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## JAIRO Cloud チェックボックスをチェック解除状態にする

チェックを解除する場合、「JAIRO Cloudを禁止しますか？」ダイアログが表示されるので、「次を入力して実行します。」で指示された文字列を入力し、「禁止」をクリックする。チェックボックスが解除状態になること

In [ ]:
async def _step(page):
    checkbox = page.locator('//input[@type = "checkbox" and @data-addon-full-name = "JAIRO Cloud"]')
    if await checkbox.is_checked():
        await checkbox.click()
        confirm_key = (await page.locator('//h4[text() = "JAIRO Cloudを禁止しますか？"]/../..//strong').inner_text()).strip()
        await page.locator('//input[@id = "wekoDeleteKey"]').fill(confirm_key)
        await page.locator('//button[text() = "禁止"]').click()
        await expect(checkbox).not_to_be_checked(timeout=transition_timeout)
    else:
        print('JAIRO Cloud already disabled for this institution')

await run_pw(_step)


## 新規タブを開き、GRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()
    await grdm.expect_anonymous_toppage(page, idp_name_2, transition_timeout=transition_timeout)

await run_pw(_step, new_page=True)

## IdPを利用し、既存ユーザー2としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードから「新規プロジェクト作成」をクリックする

「新規プロジェクトの作成」のダイアログが作成されること

In [ ]:
async def _step(page):
    trigger = page.locator('//*[@data-test-create-project-modal-button]')
    await expect(trigger).to_be_visible(timeout=transition_timeout)
    await trigger.click()
    await expect(page.locator('//input[@data-test-new-project-title]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## タイトル「TEST-WEKO-YYYYMMDDHHMM(実施時刻)」でプロジェクトを作成する

ダッシュボードのプロジェクト一覧に入力したプロジェクト名のプロジェクトが追加されること

In [ ]:
async def _step(page):
    await page.locator('//input[@data-test-new-project-title]').fill(project_name)
    await page.locator('//button[text() = "作成"]').click()
    await expect(page.locator('//button[@data-test-stay-here]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[@data-test-stay-here]').click()
    await expect(page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

作成したプロジェクトのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    project_link = page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]')
    await project_link.click()
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    global project_url
    project_url = page.url

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

アドオン設定画面が表示されるが、JAIRO Cloudアドオンは存在していないこと

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)
    addon_present = await page.locator('//div[@full_name = "JAIRO Cloud"]').count()
    if addon_present:
        raise AssertionError('JAIRO Cloud addon should not be available before enabling it for the institution')

await run_pw(_step)


## 統合管理者画面のタブに戻り、JAIRO Cloud チェックボックスをチェック状態にする

チェックボックスがチェック状態になること

In [ ]:
await close_latest_page()

async def _step(page):
    checkbox = page.locator('//input[@type = "checkbox" and @data-addon-full-name = "JAIRO Cloud"]')
    if not await checkbox.is_checked():
        await checkbox.click()
        await expect(checkbox).to_be_checked(timeout=transition_timeout)
    else:
        print('JAIRO Cloud already enabled for this institution')

await run_pw(_step)


## (必要に応じて)JAIRO CloudのOAuthアプリケーションがあれば、「アプリケーションの削除」をクリックする

OAuthアプリケーションが空になること

In [ ]:
async def _step(page):
    if not clear_apps:
        # 削除しない
        return
    locator = page.locator('//a[@href = "#wekoDisconnectAccount"]')
    while await locator.count() > 0:
        await locator.nth(0).click()
        await page.locator('//button[text() = "切断"]').click()

await run_pw(_step)

In [ ]:
client_id = None
client_secret = None


## WEKOのURLを開く

WEKO3のトップ画面が表示されること

In [ ]:
async def _step(page):
    await page.goto(weko_url)
    await expect(page.locator('//a[contains(@class, "login-button")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step, new_page=True)


## 「ログイン」をクリックする

WEKO3のログイン画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(@class, "login-button")]').click()
    await expect(page.locator('//input[@name = "email"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## WEKOアカウントを入力し、「Log in」をクリックする

WEKO3のアイテム画面が表示されること


In [ ]:
async def _step(page):
    await page.locator('//input[@name = "email"]').fill(weko_admin_email)
    await page.locator('//input[@name = "password"]').fill(weko_admin_password)
    await page.locator('//button[@type = "submit"]').click()
    await expect(page.locator('//a[@href = "/account/settings/profile/" and .//i[@class="fa fa-user"]]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 上部ナビゲーションバーの「▼」をクリックする

ドロップダウンが表示されること

In [ ]:
async def _step(page):
    await page.locator('//div[contains(@class, "navbar-form")]//button[contains(@class, "dropdown-toggle")]').click()
    await expect(page.locator('//a[@href = "/account/settings/applications/"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「Applications」をクリックする

Developer Applicationsが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/account/settings/applications/"]').click()
    await expect(page.locator('//strong[contains(text(), "Developer Applications")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「New application」をクリックする

New OAuth Applicationフォームが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "New application")]').click()
    await expect(page.locator('//strong[contains(text(), "New OAuth Application")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## アプリケーション情報を入力し、「登録」をクリックする

フォームには以下の情報を入力する。Client IDとClient Secretが表示されること

- 名前: TEST-WEKO-APP-YYYYMMDDHHMM(実施時刻)
- 記述: TEST-WEKO-APP-YYYYMMDDHHMM(実施時刻)
- Website URL: (GakuNin RDMシステムのURL)
- Redirect URIs: (GakuNin RDMシステムのURL)/oauth/callback/weko/(WEKO3リポジトリのホスト名: 例えばweko3.rdm.nii.ac.jp)/
- Client type: Confidential

In [ ]:
base_url = rdm_url[:-1] if rdm_url.endswith('/') else rdm_url
redirect_uri = f"{base_url}/oauth/callback/weko/{urlparse(weko_url).hostname}/"
application_description = oauth_application_name
print('Redirect URI:', redirect_uri)


In [ ]:
async def _step(page):
    await page.locator('//input[@name = "name"]').fill(oauth_application_name)
    await page.locator('//textarea[@name = "description"]').fill(application_description)
    # localhost:port は websiteに指定できないので置き換え
    website_url = rdm_url.replace('localhost', 'example.com')
    await page.locator('//input[@name = "website"]').fill(website_url)
    await page.locator('//textarea[@name = "redirect_uris"]').fill(redirect_uri)
    await page.locator('//button[@type = "submit"]').click()
    await expect(page.locator('//strong[text() = "Client ID"]')).to_be_visible(timeout=transition_timeout)
    global client_id, client_secret
    client_id = (await page.locator('//strong[text() = "Client ID"]/../code').inner_text()).strip()
    client_secret = (await page.locator('//strong[text() = "Client Secret"]/../code').inner_text()).strip()
    print('Retrieved client credentials from WEKO')

await run_pw(_step)


## 統合管理者画面のタブに戻り、JAIRO Cloudの「追加」をクリックする

「JAIRO Cloudアプリケーションの設定」ダイアログが表示されること

In [ ]:
await close_latest_page()

async def _step(page):
    await page.locator('//button[@href = "#wekoInputHost"]').click()
    await expect(page.locator('//h3[text() = "JAIRO Cloudアプリケーションの設定"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## アプリケーション情報を入力し、「保存」をクリックする

アプリケーション情報は以下を入力する。OAuthアプリケーションに TEST-WEKO-APP-YYYYMMDDHHMM が表示されること

- JAIRO Cloud表示名: TEST-WEKO-APP-YYYYMMDDHHMM(実施時刻)
- JAIRO Cloud URL: WEKO3リポジトリのURL
- JAIRO Cloud OAuthクライアントID: WEKO3で発行されたClient ID
- JAIRO Cloud OAuthクライアントシークレット: WEKO3で発行されたClient Secret

In [ ]:
async def _step(page):
    await page.locator('//input[@name = "weko_name"]').fill(oauth_application_name)
    await page.locator('//input[@name = "weko_url"]').fill(weko_url)
    await page.locator('//input[@name = "weko_oauth_client_id"]').fill(client_id)
    await page.locator('//input[@name = "weko_oauth_client_secret"]').fill(client_secret)
    await page.locator('//input[@name = "weko_oauth_client_secret"]').press('Tab')
    await page.locator('//button[@data-bind = "click: addHost, enable: hostCompleted"]').click()
    await expect(page.locator(f'//em[text() = "{oauth_application_name}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 統合管理者画面のタブを閉じる

統合管理者画面での設定が完了したら、このタブを閉じてユーザー側のブラウザに戻る。

In [ ]:
async def _step(page):
    await page.goto(project_url)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

後始末

In [ ]:
import json
import os

# Save OAuth client info for use by coordinator notebook
output_file = os.path.join(default_result_path, 'weko_oauth_client.json')
with open(output_file, 'w') as f:
    json.dump({
        'client_id': client_id,
        'client_secret': client_secret,
        'oauth_application_name': oauth_application_name,
    }, f)
print(f'Saved OAuth client info to {output_file}')

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)


In [ ]:
!rm -fr {work_dir}
